[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Peewee, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)

# Relationships


## What you will be able to do

Read a `ForeignKeyField` as the three things it makes: a column, a constraint and two attributes.
Say what `book.author`, `book.author_id` and `author.books` each cost in queries, and prove it with
`count_queries` rather than guessing. Write a join that actually fetches the other table, and
recognize the one that adds a join to the SQL and still asks again for every row. Use `objects()`,
and know which column it quietly loses. Switch on the foreign key enforcement that SQLite leaves off,
and see the difference in what the database will accept. Delete a parent safely, two ways. Write a
many to many relationship and create the table it needs.


## The idea

### The problem

`Book.select().join(Author)` reads like it fetches books and their authors. It does not. `join` adds
the `INNER JOIN` to the SQL and selects no columns from the joined table, so every `book.author` in
the loop that follows is a fresh query. Three books, four queries. Ten thousand books, ten thousand
and one.

Nothing about it looks wrong. The join is there in the SQL, the results are correct, and the only
symptom is time. It is the most common performance bug in every object-relational mapper, and in
peewee it has a precise cure that is one argument long.

### What a foreign key is

`author = ForeignKeyField(Author, backref="books")` on `Book` creates:

- a column, `author_id`, holding the author's key
- a constraint in the table, saying that value must exist in `author`
- `book.author`, which gives an `Author` object, fetching it if it has to
- `author.books`, from the `backref`, which is a query for that author's books

The first two are the database's. The last two are peewee's, and they are where the queries come
from.

### Why it works that way

`book.author` is a descriptor. When the row was loaded without the author's columns, the descriptor
has only an integer, so the only way it can return an `Author` is to go and get one. It cannot know
whether it is being read once or inside a loop over ten thousand rows, and it cannot refuse. Being
helpful one row at a time is the whole problem.

### Where this shows up

Any list that shows something from a related row: books with their authors, orders with their
customers, comments with their posts. It is invisible on the twelve rows you develop against and
obvious on the hundred thousand in production.

### What this notebook covers

The foreign key and its two attributes, with the cost of each one measured. The join that fetches
nothing, and the select that fixes it. `objects()`, what it flattens and what it loses. Foreign key
enforcement, which SQLite leaves switched off, and the `db` line the rest of this guide uses.
Deleting a parent, in the database and in Python. Many to many, and the table it needs. Then the
four failures, two of them silent.

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
from peewee import CharField, ForeignKeyField, Model, SqliteDatabase
from playhouse.test_utils import assert_query_count

db = SqliteDatabase(":memory:")


class Author(Model):
    name = CharField()

    class Meta:
        database = db


class Book(Model):
    title = CharField()
    author = ForeignKeyField(Author, backref="books")

    class Meta:
        database = db


db.create_tables([Author, Book])
for name, title in [("Ursula Vance", "The Salt Road"), ("Marco Pietra", "Stone and Tide"),
                    ("Kofi Mensah", "Harmattan")]:
    Book.create(title=title, author=Author.create(name=name))

try:
    with assert_query_count(1):                     # three books, one query, surely
        for book in Book.select().join(Author):
            book.author.name
except AssertionError as error:
    print("Book.select().join(Author)          -> assert_query_count says", error)

with assert_query_count(1):
    for book in Book.select(Book, Author).join(Author):
        book.author.name
print("Book.select(Book, Author).join(...) -> 1, as asserted")
```

```
Book.select().join(Author)          -> assert_query_count says 4 != 1
Book.select(Book, Author).join(...) -> 1, as asserted
```

One query for the books and one for each author's row: four. The fix is telling `select` which
tables you want columns from, and the difference between the two lines is those five characters.
`assert_query_count` is what turns "this is slow" into a test that fails.


## Setup

Six imports, peewee installed and pinned, the catalog's models, and foreign keys switched on.

- `peewee` is the library, and `Model`, the field classes and `SqliteDatabase`, from it, are what a
  model is written with
- `ManyToManyField` is the last relationship in this notebook, and `IntegrityError` and
  `OperationalError` are what the database raises when a relationship is not satisfied
- `assert_query_count` and `count_queries`, from `playhouse.test_utils`, count the queries a block
  sends, which is how every claim about cost below is checked
- `subprocess`, `sys`, `version` and `PackageNotFoundError` install peewee 4.5.1 where the version is
  not that, as on Colab, whose 4.4.0 words some of these messages differently
- `AUTHORS` and `BOOKS` are the catalog, and `build` makes the tables and loads them
- `sql` prints the SQL a query will send, with the values that go beside it

The `db` line has changed from the notebooks before this one. `SqliteDatabase(":memory:",
pragmas={"foreign_keys": 1})` is what the rest of this guide uses, because SQLite does not enforce
foreign keys unless it is told to, per connection, every time. That is the subject of one of the
worked examples, and the reason the change is worth making once rather than remembering each time.


In [1]:
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

try:
    if version("peewee") != "4.5.1":                                # Colab has 4.4.0, whose wording differs
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "peewee==4.5.1"], check=True)

import peewee
from peewee import (CharField, ForeignKeyField, IntegerField, IntegrityError, ManyToManyField,
                    Model, OperationalError, SqliteDatabase)
from playhouse.test_utils import assert_query_count, count_queries

AUTHORS = [                                                         # name, the year of the first book
    ("Ursula Vance", 2014),
    ("Marco Pietra", 2009),
    ("Ines O'Brien", 1998),
    ("Kofi Mensah", 2015),
]

BOOKS = [                                                           # title, author, year, pages
    ("The Salt Road", "Ursula Vance", 2014, 312),
    ("Nightjar", "Ursula Vance", 2018, 244),
    ("The Quiet Engine", "Ursula Vance", 2021, 398),
    ("Stone and Tide", "Marco Pietra", 2009, 501),
    ("The Lantern Keeper", "Marco Pietra", 2016, 276),
    ("Riverwork", "Marco Pietra", 2022, 189),
    ("A Careful Fire", "Ines O'Brien", 1998, 420),
    ("The Long Field", "Ines O'Brien", 2004, 355),
    ("Winter Harbour", "Ines O'Brien", 2011, 263),
    ("The Drum Line", "Kofi Mensah", 2015, 198),
    ("Harmattan", "Kofi Mensah", 2019, 331),
    ("Small Machines", "Kofi Mensah", 2023, 287),
]

def sql(query):
    """The SQL a query will send, and the values that go with it, on one line."""
    statement, values = query.sql()
    return " ".join(statement.split()) + (f"  {values}" if values else "")

db = SqliteDatabase(":memory:", pragmas={"foreign_keys": 1})        # SQLite enforces nothing without this


class CatalogModel(Model):
    """Every model in the catalog names the database once, here."""

    class Meta:
        database = db


class Author(CatalogModel):
    name = CharField(max_length=60, unique=True)
    first_book = IntegerField()


class Book(CatalogModel):
    title = CharField(max_length=80)
    author = ForeignKeyField(Author, backref="books")
    year = IntegerField(index=True)
    pages = IntegerField()

def build(database):
    """Create the tables and load the catalog, in one transaction."""
    database.create_tables([Author, Book])
    with database.atomic():
        Author.insert_many([{"name": name, "first_book": year} for name, year in AUTHORS]).execute()
        written = {author.name: author.id for author in Author.select()}
        Book.insert_many([{"title": title, "author": written[author], "year": year, "pages": pages}
                          for title, author, year, pages in BOOKS]).execute()


build(db)
print("peewee", peewee.__version__, "| foreign keys on:",
      db.execute_sql("PRAGMA foreign_keys").fetchone()[0] == 1,
      "|", Author.select().count(), "authors and", Book.select().count(), "books")


peewee 4.5.1 | foreign keys on: True | 4 authors and 12 books


## Worked examples

### One field, two attributes

`book.author` is an object, `book.author_id` is the number in the column, and `author.books` is a
query:


In [2]:
book = Book.get(Book.title == "Nightjar")
author = Author.get(Author.name == "Ines O'Brien")

print("book.author     ->", type(book.author).__name__, "|", book.author.name)
print("book.author_id  ->", type(book.author_id).__name__, "|", book.author_id)
print("author.books    ->", type(author.books).__name__)
print("                  ", [written.title for written in author.books])


book.author     -> Author | Ursula Vance
book.author_id  -> int | 1
author.books    -> ModelSelect
                   ['A Careful Fire', 'The Long Field', 'Winter Harbour']


The `backref` name is the only reason `author.books` exists. It is an ordinary query, so everything
from the **Selecting Rows** notebook works on it:


In [3]:
recent = author.books.where(Book.year >= 2004).order_by(Book.year)
print(sql(recent))
print([written.title for written in recent])


SELECT "t1"."id", "t1"."title", "t1"."author_id", "t1"."year", "t1"."pages" FROM "book" AS "t1" WHERE (("t1"."author_id" = ?) AND ("t1"."year" >= ?)) ORDER BY "t1"."year"  [3, 2004]
['The Long Field', 'Winter Harbour']


### What each attribute costs

The number in the column is free, because it was already in the row. The object is not:


In [4]:
fresh = Book.get(Book.title == "Nightjar")                          # loaded without its author

with count_queries() as counter:
    fresh.author_id
print("reading author_id:", counter.count, "queries")   # the number was already on the row

fresh = Book.get(Book.title == "Nightjar")
with count_queries() as counter:
    fresh.author
    fresh.author
    fresh.author
print("reading author three times:", counter.count, "query")


reading author_id: 0 queries
reading author three times: 1 query


One query, not three: peewee keeps the author on the instance once it has fetched it. That is why
the problem is a loop over many rows rather than a line that reads `.author` twice.

### The join that fetches nothing

Here is the cost across a whole loop, measured rather than asserted:


In [5]:
def cost(label, query):
    """Run the loop and report how many queries it took."""
    with count_queries() as counter:
        for row in query:
            row.author.name
    print(f"  {label:<34} {counter.count} {'query' if counter.count == 1 else 'queries'}")


cost("select().join(Author)", Book.select().join(Author))
cost("select(Book, Author).join(Author)", Book.select(Book, Author).join(Author))
print()
print("what join() alone selects:")
print(" ", Book.select().join(Author).sql()[0].split(" FROM")[0])


  select().join(Author)              13 queries
  select(Book, Author).join(Author)  1 query

what join() alone selects:
  SELECT "t1"."id", "t1"."title", "t1"."author_id", "t1"."year", "t1"."pages"


Thirteen against one. The `SELECT` list is the evidence: `join` put the `INNER JOIN` into the SQL and
took not one column from `author`, so there was nothing on the row for `.author` to use.

`select(Book, Author)` asks for the columns of both models, and peewee then builds the `Author` from
the columns it already has:


In [6]:
query = Book.select(Book, Author).join(Author)
print(query.sql()[0].split(" FROM")[0])

with assert_query_count(1):
    listing = [(row.title, row.author.name) for row in query]
print(listing[:3])


SELECT "t1"."id", "t1"."title", "t1"."author_id", "t1"."year", "t1"."pages", "t2"."id", "t2"."name", "t2"."first_book"
[('The Salt Road', 'Ursula Vance'), ('Nightjar', 'Ursula Vance'), ('The Quiet Engine', 'Ursula Vance')]


### objects, and the column it loses

`objects()` puts every selected column onto the top model as a plain attribute, rather than building
the second model at all:


In [7]:
row = Book.select(Book, Author).join(Author).objects().first()

print("type:", type(row).__name__)
print("title:", row.title, "| name:", row.name, "| author_id:", row.author_id)


type: Book
title: The Salt Road | name: Ursula Vance | author_id: 1


`row.name` is the author's name sitting on a `Book`, which is convenient when the names do not
overlap. When they do, one of them is lost, and the one you keep is the one selected first:


In [8]:
class Imprint(Model):
    name = CharField(max_length=40)

    class Meta:
        database = db


class Series(Model):
    name = CharField(max_length=40)                                 # a name on both models
    imprint = ForeignKeyField(Imprint, backref="series")

    class Meta:
        database = db


db.create_tables([Imprint, Series])
Series.create(name="The Tide Books", imprint=Imprint.create(name="Northwind Press"))

for label, query in (("select(Series, Imprint)", Series.select(Series, Imprint)),
                     ("select(Imprint, Series)", Series.select(Imprint, Series))):
    print(f"  {label:<26} name is {query.join(Imprint).objects().first().name!r}")


  select(Series, Imprint)    name is 'The Tide Books'
  select(Imprint, Series)    name is 'Northwind Press'


The same query, the same join, two different answers, decided by the order the models were listed in
`select`. Nothing was raised and no column was reported missing. `alias` is how you keep both:


In [9]:
both = Series.select(Series, Imprint.name.alias("imprint_name")).join(Imprint).objects().first()
print("series:", both.name, "| imprint:", both.imprint_name)


series: The Tide Books | imprint: Northwind Press


There is a second trap in `objects()`. Because it never built the `Author`, the `author` attribute is
still the lazy descriptor, so touching it brings the whole problem back:


In [10]:
with count_queries() as counter:
    for row in Book.select(Book, Author).join(Author).objects():
        row.name                                                    # flattened, free
print("reading the flattened name:", counter.count, "query")

with count_queries() as counter:
    for row in Book.select(Book, Author).join(Author).objects():
        row.author                                                  # the descriptor, not flattened
print("reading .author instead:   ", counter.count, "queries")


reading the flattened name: 1 query
reading .author instead:    13 queries


`objects()` is worth reaching for when you want plain rows and are going to read the flattened names.
It is a mistake to reach for it as a cure for the N plus one, because the cure is `select(Book,
Author)`, and `objects()` on its own undoes it the moment anybody writes `.author`.

### The constraint nobody was enforcing

SQLite has foreign key enforcement switched off by default, for compatibility with databases written
before it had any. A plain `SqliteDatabase` inherits that:


In [11]:
loose = SqliteDatabase(":memory:")                                  # no pragmas


class Writer(Model):
    name = CharField(max_length=60)

    class Meta:
        database = loose


class Title(Model):
    heading = CharField(max_length=80)
    writer = ForeignKeyField(Writer, backref="titles", on_delete="CASCADE")

    class Meta:
        database = loose


loose.create_tables([Writer, Title])
real = Writer.create(name="A Real Writer")
orphan = Title.create(heading="Nobody wrote this", writer=999)      # there is no writer 999

print("PRAGMA foreign_keys:", loose.execute_sql("PRAGMA foreign_keys").fetchone()[0])
print("the orphan was written, with writer_id:", orphan.writer_id)
print("writers that exist:", [w.id for w in Writer.select()])

real.delete_instance()                                              # on_delete=CASCADE, in theory
print("titles left after deleting their writer:", Title.select().count())


PRAGMA foreign_keys: 0
the orphan was written, with writer_id: 999
writers that exist: [1]
titles left after deleting their writer: 1


There is no author 999. The constraint is in the `CREATE TABLE`, and the database is simply not
checking it, so `on_delete="CASCADE"` does nothing either. The database this notebook's `db` was
built with does check:


In [12]:
print("PRAGMA foreign_keys:", db.execute_sql("PRAGMA foreign_keys").fetchone()[0])
try:
    Book.create(title="Nobody wrote this", author=999, year=2024, pages=100)
except IntegrityError as error:
    print("peewee.IntegrityError:", error)


PRAGMA foreign_keys: 1
peewee.IntegrityError: FOREIGN KEY constraint failed


The pragma is per connection, which is why it belongs in the `SqliteDatabase(...)` call rather than
in a statement you run once: peewee reapplies it every time it opens a connection. PostgreSQL
enforces foreign keys with nothing to switch on, which is one of the differences the **SQLite and
PostgreSQL** notebook is about.

### Deleting a parent

With the pragma on, `on_delete` is the database's answer:


In [13]:
class Chapter(Model):
    heading = CharField(max_length=60)
    book = ForeignKeyField(Book, backref="chapters", on_delete="CASCADE")

    class Meta:
        database = db


db.create_tables([Chapter])
doomed = Book.create(title="A Book With Chapters", author=author, year=2024, pages=100)
for heading in ("One", "Two", "Three"):
    Chapter.create(heading=heading, book=doomed)

print("chapters before:", Chapter.select().count())
doomed.delete_instance()
print("chapters after deleting the book:", Chapter.select().count())


chapters before: 3
chapters after deleting the book: 0


The database removed them, without peewee sending a `DELETE` for any of them. `recursive=True` is the
Python side of the same job, for a database that will not do it or a relationship without a cascade:


In [14]:
kept = Book.create(title="Another Book With Chapters", author=author, year=2024, pages=100)
for heading in ("One", "Two"):
    Chapter.create(heading=heading, book=kept)

with count_queries() as counter:
    kept.delete_instance(recursive=True)
print("chapters left:", Chapter.select().count(), "| queries sent:", counter.count)


chapters left: 0 | queries sent: 2


More queries, because peewee is finding and deleting the children itself, and it works on a database
that is enforcing nothing.

### Many to many

A book has one author here, but a book has many tags and a tag has many books. That needs a third
table, and `ManyToManyField` writes it for you:


In [15]:
class Topic(Model):
    name = CharField(max_length=40)

    class Meta:
        database = db


class Shelf(Model):
    name = CharField(max_length=40)
    topics = ManyToManyField(Topic, backref="shelves")

    class Meta:
        database = db


through = Shelf.topics.get_through_model()
print("the third model:", through.__name__, "| its table:", through._meta.table_name)
print("its columns:", [field.column_name for field in through._meta.sorted_fields])


the third model: ShelfTopicThrough | its table: shelf_topic_through
its columns: ['id', 'shelf_id', 'topic_id']


That model has to be created like any other, and it is easy to leave out of `create_tables`, which
is one of the Common errors below. With it created, `add` and `remove` write and delete rows in it:


In [16]:
db.create_tables([Topic, Shelf, through])

fiction = Shelf.create(name="Fiction")
for name in ("Sea stories", "Family", "Translation"):
    fiction.topics.add(Topic.create(name=name))

print("topics on the shelf:", sorted(topic.name for topic in fiction.topics))
print("rows in the third table:", through.select().count())

print("shelves a topic sits on:",
      [shelf.name for shelf in Topic.get(Topic.name == "Sea stories").shelves])

fiction.topics.remove(Topic.get(Topic.name == "Family"))
print("after removing one:", sorted(topic.name for topic in fiction.topics),
      "| rows now:", through.select().count())


topics on the shelf: ['Family', 'Sea stories', 'Translation']
rows in the third table: 3
shelves a topic sits on: ['Fiction']
after removing one: ['Sea stories', 'Translation'] | rows now: 2


### When to reach for which

| What you want | How to write it |
|---|---|
| the key, as a number | `book.author_id` |
| the related object, one row | `book.author` |
| the related objects, many rows | `author.books`, which is a query |
| a list of rows with their parent, one query | `.select(Book, Author).join(Author)` |
| a filter on the other table | `.join(Author).where(Author.name == ...)` |
| plain rows with the columns flattened | `.objects()`, with `alias` for any shared name |
| the database to remove the children | `on_delete="CASCADE"` and the foreign key pragma |
| peewee to remove the children | `parent.delete_instance(recursive=True)` |
| many on both sides | `ManyToManyField`, and its through model in `create_tables` |

`select(Book, Author).join(Author)` is the default for any list that shows something from both
tables. Use plain `join` when you only need it for a `where`, and nothing from the joined table is
read afterwards.

### A listing, finished

The catalog as a reader would see it, in one query, with the count asserted so that the claim cannot
rot:


In [17]:
def listing(since=2000):
    """Every book from a year on, with its author, in one query."""
    query = (Book.select(Book, Author)
                 .join(Author)
                 .where(Book.year >= since)
                 .order_by(Author.name, Book.year))
    return [f"{row.author.name}: {row.title} ({row.year})" for row in query]


with assert_query_count(1):
    lines = listing()
for line in lines[:4]:
    print("  ", line)
print("  ...", len(lines), "books, in 1 query")


   Ines O'Brien: The Long Field (2004)
   Ines O'Brien: Winter Harbour (2011)
   Kofi Mensah: The Drum Line (2015)
   Kofi Mensah: Harmattan (2019)
  ... 11 books, in 1 query


The `order_by` reaches into the joined table, which the plain `join` would also have allowed. What
`select(Book, Author)` adds is that `row.author.name` in the loop costs nothing.

### Where each part came from

| In the listing | What it relies on | The section that showed it |
|---|---|---|
| `.select(Book, Author)` | columns from both tables in one row | The join that fetches nothing |
| `.join(Author)` | the `INNER JOIN` itself | One field, two attributes |
| `.where(Book.year >= since)` | a filter with its own parentheses | **Selecting Rows** |
| `.order_by(Author.name, Book.year)` | an order across both tables | **Selecting Rows** |
| `row.author.name` costing nothing | the author built from columns already fetched | The join that fetches nothing |
| `assert_query_count(1)` | the claim written as a failure | What each attribute costs |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/peewee-deep-dive/06-relationships-solutions.ipynb).

**1.** Print every book with its author's name, and use `assert_query_count` to prove it took one
query.


In [18]:
# your code here


**2.** Count the queries the same loop takes with a plain `join`, and say where the extra ones came
from.


In [19]:
# your code here


**3.** Print one author's books through the `backref`, filtered to books over 300 pages, and print
the SQL that `backref` query sends.


In [20]:
# your code here


**4.** Show that the catalog's database refuses a book whose author does not exist, and that a
database built without the pragma accepts one.


In [21]:
# your code here


**5.** Use `objects()` on a join of `Book` and `Author` and print the flattened author name, then
show what `alias` changes if you want it under a name of your own.


In [22]:
# your code here


**6.** Give a shelf two topics with `ManyToManyField`, print the rows in the through table, and then
print the shelves a topic belongs to.


In [23]:
# your code here


## Common errors

### AssertionError: 13 != 1


In [24]:
with assert_query_count(1):
    for row in Book.select().join(Author):
        row.author.name


AssertionError: 13 != 1

Twelve books and one query for the list, so thirteen. The join is in the SQL and no column of
`author` is in the `SELECT`, so every `.author` is a round trip.

This is worth writing as a test rather than watching for, because the symptom is only slowness and
the number grows with the data. `assert_query_count` around the code that builds a page is how the
count stops being something anybody has to remember:


In [25]:
with assert_query_count(1):
    for row in Book.select(Book, Author).join(Author):
        row.author.name
print("one query, and the test now guards it")


one query, and the test now guards it


### No error, and a book by an author who does not exist: foreign keys left switched off


In [26]:
unchecked = SqliteDatabase(":memory:")                              # the default, with no pragmas
checked = SqliteDatabase(":memory:", pragmas={"foreign_keys": 1})

for database in (unchecked, checked):
    database.bind([Writer, Title])                                  # the same models, another database
    database.create_tables([Writer, Title])
    setting = database.execute_sql("PRAGMA foreign_keys").fetchone()[0]
    try:
        Title.create(heading="By Nobody", writer=4321)
        print(f"  foreign_keys={setting} -> written, and the row is wrong")
    except IntegrityError as error:
        print(f"  foreign_keys={setting} -> peewee.IntegrityError: {error}")


  foreign_keys=0 -> written, and the row is wrong
  foreign_keys=1 -> peewee.IntegrityError: FOREIGN KEY constraint failed


The same two models, the same insert, two databases that differ in one setting. Every row inserted
while it was off stays wrong after it is switched on, because the pragma decides what is checked
from now on rather than what is already there.

`bind` is what let both databases use one pair of models, which is worth knowing on its own: a model
is not married to the database its `Meta` named, and tests often rebind to a database of their own.
The setting belongs on the `Database` object, where every connection it opens will carry it.


In [27]:
checked.bind([Writer, Title])
Writer.create(name="Someone Real")
kept = Title.create(heading="A Real Title", writer=Writer.get(Writer.name == "Someone Real"))

print("a title with a writer that exists:", kept.heading, "| writer_id:", kept.writer_id)
print("titles in the checked database:", Title.select().count())


a title with a writer that exists: A Real Title | writer_id: 1
titles in the checked database: 1


### No error, and the joined name missing: objects() with a name on both models


In [28]:
one = Series.select(Series, Imprint).join(Imprint).objects().first()
other = Series.select(Imprint, Series).join(Imprint).objects().first()

print("select(Series, Imprint) -> name:", repr(one.name))
print("select(Imprint, Series) -> name:", repr(other.name))
print("the two real values:    ", repr(Series.get().name), "and", repr(Imprint.get().name))


select(Series, Imprint) -> name: 'The Tide Books'
select(Imprint, Series) -> name: 'Northwind Press'
the two real values:     'The Tide Books' and 'Northwind Press'


Two rows with one `name` each, from a query that selected two. `objects()` writes every column onto
the row by its name, and the first one written is the one that stays, so the order of the models in
`select` silently decides which value you get.

Name the column you want, and the question does not arise:


In [29]:
named = (Series.select(Series.name.alias("series_name"), Imprint.name.alias("imprint_name"))
               .join(Imprint).objects().first())
print("series:", named.series_name, "| imprint:", named.imprint_name)


series: The Tide Books | imprint: Northwind Press


### peewee.OperationalError: no such table: article_label_through


In [30]:
class Label(Model):
    name = CharField(max_length=40)

    class Meta:
        database = db


class Article(Model):
    heading = CharField(max_length=60)
    labels = ManyToManyField(Label, backref="articles")

    class Meta:
        database = db


db.create_tables([Article, Label])                                  # the through model is missing
Article.create(heading="A piece").labels.add(Label.create(name="news"))


OperationalError: no such table: article_label_through

`ManyToManyField` is not a column. It is a third model, made for you and named after the two it
joins, and `create_tables` only makes the models you hand it. Listing the two obvious models is
exactly the mistake, because the field is written inside one of them and looks like part of it.

`get_through_model()` is how you get hold of it:


In [31]:
db.create_tables([Article.labels.get_through_model()])
piece = Article.get(Article.heading == "A piece")
piece.labels.add(Label.get(Label.name == "news"))

print("labels:", [label.name for label in piece.labels])
print("articles for the label:", [a.heading for a in Label.get(Label.name == "news").articles])


labels: ['news']
articles for the label: ['A piece']


## Recap

- A `ForeignKeyField` makes a column, a constraint and two attributes: `book.author` and, from the
  `backref`, `author.books`.
- `book.author_id` is free. `book.author` costs a query if the row was loaded without the author's
  columns, and is then kept on the instance.
- `join` adds the `INNER JOIN` and selects nothing from the joined table, so a loop over the result
  sends one query per row.
- `select(Book, Author).join(Author)` selects both tables' columns and costs one query for the lot.
- `assert_query_count` and `count_queries` turn a claim about cost into something that can fail.
- `objects()` flattens every selected column onto the top model. Where two models have a column of
  the same name the first one selected wins and the other is lost, and `alias` is the answer.
  Reading a foreign key attribute on an `objects()` row brings back the query per row.
- SQLite does not enforce foreign keys unless told, per connection, so the guide builds every
  database with `pragmas={"foreign_keys": 1}`. Without it, `on_delete` does nothing.
- `delete_instance(recursive=True)` is the Python side of a cascade, and costs more queries.
- `ManyToManyField` makes a third model that has to be created, which `get_through_model()` returns.


## What is next

The **prefetch and Load** notebook is about the relationship a join cannot flatten: the many side.
One author with three books is three rows of a join and one author in Python, so `prefetch` fetches
each level with its own query and assembles them, and `Load` and `per_parent` decide how much of
each level comes back.


---

&#8592; **Previous:** [Transactions](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/peewee-deep-dive/05-transactions.ipynb)  &nbsp;·&nbsp;  [Peewee, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)
